In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ====== 3. Install Python Packages ======
# !pip install torch torchvision torchaudio --upgrade  # PyTorch
!pip install opencv-python matplotlib numpy pycocotools psutil Pillow
!pip install tqdm

# ====== 4. Verify GPU Availability (if using CUDA) ======
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/CV Project 2025/segmentation')

In [4]:
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone


from coco_utils import get_coco, collate_fn
from engine import train_one_epoch, evaluate

In [5]:
def get_instance_segmentation_model(num_classes):
    # load an instance segmentation model pre-trained on COCO
    backbone = resnet_fpn_backbone('resnet101', pretrained=True)
    model = MaskRCNN(backbone, num_classes)

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model


In [6]:
img_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/images'
train_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/train.json'
val_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/val.json'
test_nn_data_path = '/content/drive/MyDrive/CV Project 2025/armbench/same-object-transfer-set/test.json'

In [7]:
coco_train_ds = get_coco(img_data_path, train_nn_data_path, mode="instances", train=True)
coco_val_ds = get_coco(img_data_path, val_nn_data_path, mode="instances", train=False)
coco_test_ds = get_coco(img_data_path, test_nn_data_path, mode="instances", train=False)

loading annotations into memory...
Done (t=0.94s)
creating index...
index created!
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
loading annotations into memory...
Done (t=0.34s)
creating index...
index created!


In [8]:
train_data_loader = torch.utils.data.DataLoader(coco_train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)

val_data_loader = torch.utils.data.DataLoader(coco_val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

test_data_loader = torch.utils.data.DataLoader(coco_test_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [9]:
num_classes = 2

# get the model using our helper function
model = get_instance_segmentation_model(num_classes)
# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# and a learning rate scheduler which decreases the learning rate by
# 10x every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# let's train it for 10 epochs
from torch.optim.lr_scheduler import StepLR

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-

In [10]:
num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 5 iterations
    train_one_epoch(model, optimizer, train_data_loader, device, epoch, print_freq=5)
    # update the learning rate
    lr_scheduler.step()
    # # evaluate on the test dataset
    evaluate(model, val_data_loader, device=device)

    torch.save(
      {
          "epoch": epoch,
          "model_state_dict": model.state_dict(),
          "optimizer_state_dict": optimizer.state_dict(),
      },
      "maskrcnn_resnet18_same_object.pt",
    )

/content/drive/MyDrive/CV Project 2025/segmentation/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/580]  eta: 1:32:38  lr: 0.000014  loss: 2.5212 (2.5212)  loss_classifier: 0.6956 (0.6956)  loss_box_reg: 0.0116 (0.0116)  loss_mask: 1.0773 (1.0773)  loss_objectness: 0.6911 (0.6911)  loss_rpn_box_reg: 0.0456 (0.0456)  time: 9.5842  data: 5.2395  max mem: 5816
Epoch: [0]  [  5/580]  eta: 0:29:03  lr: 0.000057  loss: 2.5212 (2.5810)  loss_classifier: 0.6911 (0.6932)  loss_box_reg: 0.0168 (0.0365)  loss_mask: 1.0773 (1.1333)  loss_objectness: 0.6901 (0.6904)  loss_rpn_box_reg: 0.0228 (0.0276)  time: 3.0324  data: 1.0396  max mem: 6063
Epoch: [0]  [ 10/580]  eta: 0:23:21  lr: 0.000100  loss: 2.4684 (2.5009)  loss_classifier: 0.6847 (0.6763)  loss_box_reg: 0.0217 (0.0302)  loss_mask: 1.0272 (1.0762)  loss_objectness: 0.6901 (0.6902)  loss_rpn_box_reg: 0.0310 (0.0280)  time: 2.4580  data: 0.6711  max mem: 6077
Epoch: [0]  [ 15/580]  eta: 0:20:48  lr: 0.000143  loss: 2.3775 (2.4052)  loss_classifier: 0.6598 (0.6443)  loss_box_reg: 0.0175 (0.0254)  loss_mask: 1.0016 (1.0100) 

In [12]:
# download the model
from google.colab import files
files.download("maskrcnn_resnet101_same_object.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes

from PIL import Image, ImageOps
import PIL
import numpy as np

def save_result(img, int_img, prediction):
    img = Image.fromarray(img.mul(255).permute(1, 2, 0).byte().numpy())

    output = prediction[0]
    masks, boxes = output["masks"], output["boxes"]

    detection_threshold = 0.8
    pred_scores = output["scores"].detach().cpu().numpy()
    pred_classes = [str(i) for i in output["labels"].cpu().numpy()]
    print(pred_classes)
    print(output["labels"])
    pred_bboxes = output["boxes"].detach().cpu().numpy()
    boxes = pred_bboxes[pred_scores >= detection_threshold].astype(np.int32)
    pred_classes = pred_classes[: len(boxes)]

    int_img = np.array(int_img)
    int_img = np.transpose(int_img, [2, 0, 1])
    int_img = torch.tensor(int_img, dtype=torch.uint8)

    colors = np.random.randint(0, 255, size=(len(pred_bboxes), 3))
    colors = [tuple(color) for color in colors]
    result_with_boxes = draw_bounding_boxes(
        int_img,
        boxes=torch.tensor(boxes),
        width=4,
        colors=colors,
        labels=pred_classes,
    )

    final_masks = masks > 0.5
    final_masks = final_masks.squeeze(1)
    seg_result = draw_segmentation_masks(
        result_with_boxes, final_masks, colors=colors, alpha=0.8
    )

    seg_img = Image.fromarray(seg_result.mul(255).permute(1, 2, 0).byte().numpy())

    imgs = [img, seg_img]
    min_shape = sorted([(np.sum(i.size), i.size) for i in imgs])[0][1]
    imgs_comb = np.hstack([i.resize(min_shape) for i in imgs])

    imgs_comb = Image.fromarray(imgs_comb)
    imgs_comb.save("resnet101_maskrcnn_result.jpg")

In [14]:
# test the model on the test set

evaluate(model, test_data_loader, device=device)

Test:  [  0/507]  eta: 0:10:36  model_time: 0.2283 (0.2283)  evaluator_time: 0.0243 (0.0243)  time: 1.2552  data: 0.9903  max mem: 6389
Test:  [100/507]  eta: 0:04:11  model_time: 0.1769 (0.2817)  evaluator_time: 0.1038 (0.2452)  time: 0.6446  data: 0.0470  max mem: 6389
Test:  [200/507]  eta: 0:03:03  model_time: 0.2442 (0.2797)  evaluator_time: 0.1569 (0.2354)  time: 0.6757  data: 0.0417  max mem: 6389
Test:  [300/507]  eta: 0:01:59  model_time: 0.2110 (0.2707)  evaluator_time: 0.1233 (0.2237)  time: 0.6019  data: 0.0597  max mem: 6389
Test:  [400/507]  eta: 0:01:01  model_time: 0.2128 (0.2719)  evaluator_time: 0.1174 (0.2242)  time: 0.5403  data: 0.0467  max mem: 6389
Test:  [500/507]  eta: 0:00:04  model_time: 0.2349 (0.2726)  evaluator_time: 0.1847 (0.2239)  time: 0.5966  data: 0.0502  max mem: 6389
Test:  [506/507]  eta: 0:00:00  model_time: 0.2136 (0.2719)  evaluator_time: 0.1027 (0.2226)  time: 0.5434  data: 0.0514  max mem: 6389
Test: Total time: 0:04:52 (0.5761 s / it)
Averag

In [18]:
# randomly sample one index from the test set
sample_idx = torch.randint(len(coco_test_ds), size=(1,)).item()
int_img, img, _ = coco_test_ds[sample_idx]
model.eval()
with torch.no_grad():
  prediction = model([img.to(device)])

save_result(img, int_img, prediction)

['1', '1', '1']
tensor([1, 1, 1], device='cuda:0')


In [19]:
# download the sample test image
from google.colab import files
files.download("resnet101_maskrcnn_result.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>